In [1]:
# Install dependencies
!pip install -q transformers peft bitsandbytes datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 20.3 MB/s eta 0:00:00


In [2]:
# Dataset
from datasets import Dataset

dataset = Dataset.from_list([
    {"text": "Q: What is AI?\nA: AI is machines acting smart."},
    {"text": "Q: What is Python?\nA: Python is a programming language."}
])

In [3]:
# Load model (4 bit)
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb = BitsAndBytesConfig(load_in_4bit=True)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb,
    device_map="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [4]:
# Add LoRA
from peft import LoraConfig, get_peft_model

lora = LoraConfig(
    r=4,
    lora_alpha=8,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora)

In [5]:
# Tokenize
def tokenize(example):
    result = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

    result["labels"] = result["input_ids"].copy()  # ✅ VERY IMPORTANT
    return result

dataset = dataset.map(tokenize, remove_columns=["text"])

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

In [6]:
# Train
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    output_dir="out",
    per_device_train_batch_size=1,
    num_train_epochs=1,
    logging_steps=1,
    fp16=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=dataset,
)

trainer.train()

Step,Training Loss
1,12.404577
2,11.591972


TrainOutput(global_step=2, training_loss=11.998274803161621, metrics={'train_runtime': 2.9319, 'train_samples_per_second': 0.682, 'train_steps_per_second': 0.682, 'total_flos': 1589876097024.0, 'train_loss': 11.998274803161621, 'epoch': 1.0})

In [8]:
# Test
input_text = "Q: What is AI?\nA:"
inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

output = model.generate(**inputs, max_new_tokens=30)
print(tokenizer.decode(output[0], skip_special_tokens=True))

Q: What is AI?
A: Artificial Intelligence (AI) is the ability of machines to perform tasks that require human intelligence. It is a field of computer science that focus
